# 03 — Del texto al registro

**Qué hace.** Extrae los registros de entrevista con dos caminos (reglas y LLM), aplica la regla de verificación literal y mide las dos capas de métricas por separado.

**Qué lee.** La caché de transcripciones (`rutas.cache_dir`, que es `data/cache/`), `data/ground_truth/entrevistas.csv` y `data/referencias/*.txt`.

**Qué produce.** Los registros extraídos y las métricas de las dos capas en `rutas.resultados_dir`.

Llama a `src.normalizacion`, `src.extraccion_reglas`, `src.extraccion_llm`, `src.verificacion` y `src.metricas`; no reimplementa nada.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config
from src.normalizacion import normalizar_texto, palabras_a_numeros

cfg = cargar_config()
rutas = cfg["rutas"]
pd.set_option("display.max_colwidth", None)

## 1. Capa de normalización

En el papel, "seis mil quinientos" y 6500 son lo mismo. En el audio no: si la transcripción dice las palabras y el registro guarda la cifra, la verificación literal falla aunque todo esté bien. Esta capa existe solo porque la entrada es voz.

Dos funciones de `src/normalizacion.py`. Las reglas completas están en el docstring del módulo y los parámetros en `config.yaml → normalizacion`:

- **`palabras_a_numeros(texto)`** convierte los números dichos en palabras a dígitos, sin tocar el resto de la frase. Usa `text2num` y añade reglas propias donde la librería se equivoca: artículos, ordinales, apócope, "y medio", dictados y códigos.
- **`normalizar_texto(texto)`** es la regla única para comparar referencia y transcripción: números a dígitos, minúsculas, sin puntuación, espacios colapsados. Las tildes se mantienen (opción `quitar_tildes`).

In [2]:
print(json.dumps(cfg["normalizacion"], ensure_ascii=False, indent=1))

{
 "minusculas": true,
 "quitar_puntuacion": true,
 "palabras_a_numeros": true,
 "quitar_tildes": false,
 "colapsar_espacios": true,
 "umbral_numeros_pequenos": 2,
 "articulos": [
  "un",
  "una",
  "uno",
  "unos",
  "unas"
 ],
 "alias": {
  "veintiún": "veintiuno",
  "veintiun": "veintiuno",
  "veintidos": "veintidós",
  "veintitres": "veintitrés",
  "veintiseis": "veintiséis"
 },
 "palabras_protegidas": [
  "por ciento",
  "primer",
  "primero",
  "primera",
  "segundo",
  "segunda",
  "tercer",
  "tercero",
  "tercera",
  "cuarto",
  "cuarta",
  "quinto",
  "quinta",
  "sexto",
  "sexta",
  "séptimo",
  "séptima",
  "octavo",
  "octava",
  "noveno",
  "novena",
  "décimo",
  "décima"
 ],
 "unidades_y_medio": [
  "año",
  "años",
  "mes",
  "meses",
  "hora",
  "horas"
 ],
 "min_digitos_dictado": 3,
 "multiplicadores": {
  "mil": 1000,
  "millón": 1000000,
  "millon": 1000000,
  "millones": 1000000
 },
 "cantidades_palabra": {
  "un": 1,
  "una": 1,
  "medio": 0.5
 },
 "codigos": [


### 1.1 Casos de `palabras_a_numeros`

Cada caso sale de los guiones. Las filas con `limitacion = True` son casos sin una buena solución en esta capa: se muestran con su salida ideal (`esperado`) y con la que de verdad se obtiene (`queda`), en vez de esconderlos. Los casos marcados *(hallado en …)* no estaban en la lista inicial: aparecieron al aplicar la función a las referencias completas (sección 1.4).

In [3]:
casos = [
    # grupo, entrada, esperado, motivo
    ("cifras", "seis mil quinientos soles", "6500 soles", "pretensión de E1"),
    ("cifras", "mil ochocientos dólares", "1800 dólares", "pretensión de E2"),
    ("cifras", "nueve mil quinientos", "9500", "pretensión ideal de E3"),
    ("direcciones", "Calle Los Cedros doscientos cuarenta y cinco", "Calle Los Cedros 245", "L1"),
    ("direcciones", "Jirón Los Halcones mil treinta y cuatro", "Jirón Los Halcones 1034", "L3"),
    ("direcciones", "Avenida Los Laureles dos mil doscientos setenta y uno", "Avenida Los Laureles 2271", "L5"),
    ("años", "desde dos mil veintitrés, o sea, tres años", "desde 2023, o sea, 3 años",
     "el año queda como cifra; pasarlo a duración es trabajo de la extracción"),
    ("años", "empecé como tester manual en dos mil veinte", "empecé como tester manual en 2020", "E5"),
    ("apócope", "su total es ciento veintiún soles", "su total es 121 soles",
     "text2num no conoce 'veintiún': alias a 'veintiuno'"),
    ("apócope", "el total es ciento treinta y un soles", "el total es 131 soles", "'y un' dentro del compuesto"),
    ("tildes", "dos mollejitas, a dieciséis cada una", "2 mollejitas, a 16 cada una", "L4"),
    ("tildes", "veintitrés", "23", "con tilde"),
    ("tildes", "veintitres", "23", "si el ASR omite la tilde: alias"),
    ("teléfono", "nueve ocho siete, seis cinco cuatro, tres dos uno", "987, 654, 321",
     "dictado por grupos: cada grupo se une, las comas separan"),
    ("artículos", "un pollo entero", "un pollo entero", "artículo: la cantidad 1 la decide la extracción"),
    ("artículos", "una jarra de chicha morada", "una jarra de chicha morada", "artículo"),
    ("artículos", "un medio pollo", "un medio pollo", "artículo + tamaño"),
    ("artículos", "dos jarras de chicha morada", "2 jarras de chicha morada", "desde 'dos' sí se convierte"),
    ("artículos", "casi un quince por ciento", "casi un 15 por ciento", "hallado en E2: text2num daba '1 15'"),
    ("artículos", "le llega en unos cuarenta minutos", "le llega en unos 40 minutos", "hallado en L2: daba '1 40'"),
    ("artículos", "a veinte soles cada uno, y tres papas", "a 20 soles cada uno, y 3 papas",
     "hallado en L3: daba 'cada 1, y 3'"),
    ("artículos", "de treinta y seis y una ensalada clásica", "de 36 y una ensalada clásica",
     "hallado en L4: tras 'seis y' es artículo, tras 'treinta y' sería número"),
    ("medio", "medio pollo", "medio pollo", "'medio' es un tamaño"),
    ("medio", "una papa familiar", "una papa familiar", "artículo + tamaño"),
    ("medio", "una gaseosa de litro y medio", "una gaseosa de litro y medio", "tamaño de la carta, no tiempo"),
    ("medio", "cuatro años y medio", "4.5 años", "número + unidad de tiempo + 'y medio'"),
    ("medio", "un año y medio", "1.5 año", "E4: el artículo también vale como 1 aquí"),
    ("ordinales", "tres cuartos de pollo", "3 cuartos de pollo", "text2num daba '3 4ᵒˢ': ordinal protegido"),
    ("ordinales", "primero de diciembre", "primero de diciembre", "la fecha la interpreta la extracción"),
    ("ordinales", "la segunda semana de enero", "la segunda semana de enero", "text2num daba '2ª'"),
    ("ordinales", "cien por ciento", "100 por ciento", "text2num daba '100 por 100'"),
    ("código", "LEÑA cinco", "LEÑA5", "código de promoción unido"),
    ("código", "Leña 5", "LEÑA5", "si el ASR ya escribe la cifra"),
    ("código", "La Leña Dorada", "La Leña Dorada", "sin número detrás no es el código"),
    ("autocorrección", "mil cuarenta y tres… no, perdón, me equivoqué, es mil treinta y cuatro",
     "1043… no, perdón, me equivoqué, es 1034", "se convierten los dos; el final lo elige la extracción"),
    ("cifras mixtas", "Estoy entre 9 y 10 mil soles", "Estoy entre 9 y 10000 soles",
     "hallado en E3 con large-v3: Whisper escribe '10 mil' y daba '10 1000'"),
    ("cifras mixtas", "3 mil soles", "3000 soles", "cifra + 'mil'"),
    ("cifras mixtas", "1.5 millones", "1500000", "punto decimal"),
    ("cifras mixtas", "1,5 millones", "1500000", "coma decimal"),
    ("cifras mixtas", "1.500 millones", "1500000000", "tres dígitos tras el punto: separador de miles"),
    ("cifras mixtas", "un millón", "1000000", "daba 'un 1000000': el artículo apartado cortaba el número"),
    ("cifras mixtas", "un millón y medio", "1500000", "daba 'un 1000000 y medio'"),
    ("cifras mixtas", "medio millón", "500000", "daba 'medio 1000000'"),
    ("cifras mixtas", "10 mil 500", "10500", "la cifra menor que sigue al multiplicador se suma"),
    ("cifras mixtas", "mil 500", "1500", "multiplicador sin cantidad delante vale 1"),
    ("cifras mixtas", "2 millones 500 mil", "2500000", "primero 500 mil, después los millones"),
    ("cifras mixtas", "dos millones quinientos mil", "2500000", "en palabras ya funcionaba: no debe cambiar"),
]

limitaciones = [
    # grupo, entrada, esperado (ideal), queda (lo que hace), motivo
    ("rango elíptico", "entre nueve y diez mil soles", "entre 9000 y 10000 soles", "entre 9 y 10000 soles",
     "el 'mil' se aplica a los dos extremos; resolverlo es interpretar el rango, no convertir palabras. "
     "En E3 no afecta: vale el ideal 9500"),
    ("medio sin número", "llevo año y medio", "llevo 1.5 año", "llevo año y medio",
     "la regla exige número o artículo delante; no aparece en el corpus"),
    ("forma literaria", "mil y una noches", "1001 noches", "1000 y una noches",
     "tras 'mil y' la regla de artículos aparta 'una'; no aparece en el corpus"),
    ("cifra pegada", "cobran 10 mil 2 veces", "cobran 10000 2 veces", "cobran 10002 veces",
     "una cifra menor que el multiplicador justo detrás se suma aunque no sea parte del número; "
     "no aparece en el corpus"),
]

tabla = pd.DataFrame(
    [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e),
      "limitacion": False, "motivo": m} for g, e, esp, m in casos]
    + [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e),
        "limitacion": True, "motivo": m} for g, e, esp, _queda, m in limitaciones]
)
tabla.insert(4, "pasa", tabla.obtenido == tabla.esperado)
print(f"{tabla.pasa.sum()} de {len(tabla)} casos pasan · {tabla.limitacion.sum()} limitaciones documentadas")
tabla

47 de 51 casos pasan · 4 limitaciones documentadas


,grupo,entrada,esperado,obtenido,pasa,limitacion,motivo
0,cifras,seis mil quinientos soles,6500 soles,6500 soles,True,False,pretensión de E1
1,cifras,mil ochocientos dólares,1800 dólares,1800 dólares,True,False,pretensión de E2
2,cifras,nueve mil quinientos,9500,9500,True,False,pretensión ideal de E3
3,direcciones,Calle Los Cedros doscientos cuarenta y cinco,Calle Los Cedros 245,Calle Los Cedros 245,True,False,L1
4,direcciones,Jirón Los Halcones mil treinta y cuatro,Jirón Los Halcones 1034,Jirón Los Halcones 1034,True,False,L3
5,direcciones,Avenida Los Laureles dos mil doscientos setenta y uno,Avenida Los Laureles 2271,Avenida Los Laureles 2271,True,False,L5
6,años,"desde dos mil veintitrés, o sea, tres años","desde 2023, o sea, 3 años","desde 2023, o sea, 3 años",True,False,el año queda como cifra; pasarlo a duración es trabajo de la extracción
7,años,empecé como tester manual en dos mil veinte,empecé como tester manual en 2020,empecé como tester manual en 2020,True,False,E5
8,apócope,su total es ciento veintiún soles,su total es 121 soles,su total es 121 soles,True,False,text2num no conoce 'veintiún': alias a 'veintiuno'
9,apócope,el total es ciento treinta y un soles,el total es 131 soles,el total es 131 soles,True,False,'y un' dentro del compuesto


### 1.2 Asserts

El notebook falla si un caso deja de dar su salida. Para las limitaciones se comprueba que siguen comportándose como está documentado: si una cambia (para bien o para mal), hay que enterarse.

In [4]:
for grupo, entrada, esperado, motivo in casos:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == esperado, f"[{grupo}] {entrada!r}: se esperaba {esperado!r}, salió {obtenido!r}"

for grupo, entrada, esperado, queda, motivo in limitaciones:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == queda, f"[{grupo}] la limitación cambió: {entrada!r} ahora da {obtenido!r}"
    assert obtenido != esperado, f"[{grupo}] ya no es limitación: muévelo a casos"

print(f"OK: {len(casos)} casos y {len(limitaciones)} limitaciones se comportan como está documentado")

OK: 47 casos y 4 limitaciones se comportan como está documentado


### 1.3 `normalizar_texto`: la referencia y la transcripción deben quedar iguales

La referencia escribe los números en palabras y Whisper suele escribirlos en cifras, con separador de miles o con `%`. Si la normalización no los iguala, el WER cuenta como error algo que el modelo oyó bien. Cada par es *(como está en el guion, como podría escribirlo el ASR)*.

In [5]:
pares = [
    # referencia, variante ASR, deben_coincidir, motivo
    ("Mi pretensión sería de seis mil quinientos soles.", "mi pretensión sería de 6.500 soles", True,
     "separador de miles"),
    ("cuatro años y medio", "4,5 años", True, "coma decimal"),
    ("casi un quince por ciento", "casi un 15%", True, "'%' -> 'por ciento'"),
    ("¿Cuál es tu pretensión salarial?", "Cuál es tu pretensión salarial", True, "puntuación y mayúsculas"),
    ("Next.js", "next.js", True, "el punto entre letras es puntuación"),
    ("nueve ocho siete, seis cinco cuatro, tres dos uno", "987 654 321", True, "dictado por grupos"),
    ("Ah, tengo un código de la aplicación: LEÑA cinco.", "Ah, tengo un código de la aplicación: Leña 5.", True,
     "código"),
    ("Estoy entre nueve y diez mil soles.", "Estoy entre 9 y 10 mil soles.", True, "cifra mixta (E3)"),
    ("un millón y medio", "1.5 millones", True, "cifra mixta con decimal"),
    ("Está bien.", "esta bien", False, "decisión: las tildes se mantienen (quitar_tildes: false)"),
    ("nueve ocho siete, seis cinco cuatro, tres dos uno", "987654321", False,
     "limitación: un teléfono sin espacios no se puede volver a partir en grupos"),
]

comparacion = pd.DataFrame(
    [{"referencia": r, "variante_asr": v, "ref_normalizada": normalizar_texto(r),
      "asr_normalizada": normalizar_texto(v), "deben_coincidir": d, "motivo": m} for r, v, d, m in pares]
)
comparacion.insert(4, "coinciden", comparacion.ref_normalizada == comparacion.asr_normalizada)
comparacion

,referencia,variante_asr,ref_normalizada,asr_normalizada,coinciden,deben_coincidir,motivo
0,Mi pretensión sería de seis mil quinientos soles.,mi pretensión sería de 6.500 soles,mi pretensión sería de 6500 soles,mi pretensión sería de 6500 soles,True,True,separador de miles
1,cuatro años y medio,"4,5 años",4.5 años,4.5 años,True,True,coma decimal
2,casi un quince por ciento,casi un 15%,casi un 15 por ciento,casi un 15 por ciento,True,True,'%' -> 'por ciento'
3,¿Cuál es tu pretensión salarial?,Cuál es tu pretensión salarial,cuál es tu pretensión salarial,cuál es tu pretensión salarial,True,True,puntuación y mayúsculas
4,Next.js,next.js,next js,next js,True,True,el punto entre letras es puntuación
5,"nueve ocho siete, seis cinco cuatro, tres dos uno",987 654 321,987 654 321,987 654 321,True,True,dictado por grupos
6,"Ah, tengo un código de la aplicación: LEÑA cinco.","Ah, tengo un código de la aplicación: Leña 5.",ah tengo un código de la aplicación leña5,ah tengo un código de la aplicación leña5,True,True,código
7,Estoy entre nueve y diez mil soles.,Estoy entre 9 y 10 mil soles.,estoy entre 9 y 10000 soles,estoy entre 9 y 10000 soles,True,True,cifra mixta (E3)
8,un millón y medio,1.5 millones,1500000,1500000,True,True,cifra mixta con decimal
9,Está bien.,esta bien,está bien,esta bien,False,False,decisión: las tildes se mantienen (quitar_tildes: false)


In [6]:
for r, v, d, m in pares:
    assert (normalizar_texto(r) == normalizar_texto(v)) == d, f"{r!r} / {v!r}: {m}"

assert normalizar_texto("Mi pretensión sería de seis mil quinientos soles.") == "mi pretensión sería de 6500 soles"
assert normalizar_texto("4,5 años") == "4.5 años"
assert normalizar_texto("1.5 litros") == "1.5 litros"
assert normalizar_texto("1.000.000") == "1000000"

# quitar_tildes es una opción de config.yaml; se prueba sin editar el archivo pasando otras reglas.
con_tildes_fuera = {**cfg["normalizacion"], "quitar_tildes": True}
assert normalizar_texto("Está bien.", con_tildes_fuera) == normalizar_texto("esta bien", con_tildes_fuera)
assert normalizar_texto("La Leña Dorada, pingüino", con_tildes_fuera) == "la leña dorada pinguino"
print("OK: normalizar_texto")

OK: normalizar_texto


### 1.4 Los números de cada referencia contra el ground truth

`palabras_a_numeros` se aplica a los 10 textos de `data/referencias/`. Por cada clip se listan, en orden, todos los números sueltos que aparecen tras convertir (no cuenta el `1` de "C1" ni el `5` de "LEÑA5"). Así se puede revisar a ojo que no haya conversiones de más; con esta revisión se encontraron los cuatro casos de artículos de la tabla 1.1.

In [7]:
referencias = {
    f.stem: f.read_text(encoding="utf-8")
    for f in sorted(rutas["referencias_dir"].glob(cfg["archivos"]["patron_referencia"].format(clip="*")))
}
numeros = pd.DataFrame(
    [{"clip": clip, "cantidad": len(n := re.findall(r"\b\d+(?:\.\d+)?\b", palabras_a_numeros(texto))),
      "numeros_encontrados": " · ".join(n)} for clip, texto in referencias.items()]
)
numeros

,clip,cantidad,numeros_encontrados
0,e1,11,2 · 5 · 4.5 · 6 · 2 · 2 · 2 · 5800 · 6500 · 2 · 6
1,e2,7,2 · 2023 · 3 · 2 · 15 · 6 · 1800
2,e3,9,9 · 10000 · 9500 · 3 · 8 · 9 · 9 · 8 · 2
3,e4,7,100 · 1.5 · 4200 · 4800 · 6 · 2 · 3
4,e5,9,2 · 4 · 2 · 11 · 6200 · 7000 · 2020 · 6 · 2
5,l1,17,68 · 14 · 2 · 2 · 12 · 10 · 245 · 302 · 245 · 302 · 987 · 654 · 321 · 2 · 96 · 40 · 50
6,l2,20,2 · 2 · 36 · 2 · 72 · 68 · 2 · 2 · 3 · 18 · 9 · 5 · 1580 · 2 · 1580 · 2 · 100 · 95 · 95 · 40
7,l3,25,3 · 3 · 3 · 20 · 3 · 7 · 3 · 3 · 4 · 2 · 2 · 9 · 1043 · 5 · 1034 · 1034 · 5 · 1034 · 5 · 3 · 3 · 3 · 2 · 121 · 45
8,l4,16,36 · 10 · 2 · 2 · 16 · 2 · 2 · 5 · 5 · 418 · 702 · 418 · 702 · 2 · 85 · 35
9,l5,15,2 · 2 · 2 · 2 · 2 · 2 · 10 · 2271 · 504 · 2271 · 504 · 2 · 2 · 131 · 131


Ahora, cada valor numérico del ground truth: ¿aparece literalmente en la referencia después de normalizar? Se busca como palabra completa (`"8"` no coincide dentro de `"1800"`). Es un adelanto de la regla de verificación, que implementará `src/verificacion.py`.

Se revisan los años de experiencia y la pretensión (Task 1), y el total dicho, el descuento, el precio de cada ítem y los números de la dirección (Task 2). Las cantidades no se revisan: una cifra como `2` aparece en casi cualquier llamada, así que encontrarla no demuestra nada.

In [8]:
entrevistas = pd.read_csv(rutas["ground_truth_entrevistas"])
llamadas = pd.read_csv(rutas["ground_truth_llamadas"])


def como_token(x):
    return f"{float(x):g}"


valores = []
for _, f in entrevistas.iterrows():
    valores += [(f["clip"], "anios_experiencia", f["anios_experiencia"]),
                (f["clip"], "pretension_salarial", f["pretension_salarial"])]
for _, f in llamadas.iterrows():
    valores.append((f["clip"], "total_dicho", f["total_dicho"]))
    if f["descuento"]:
        valores.append((f["clip"], "descuento", f["descuento"]))
    for item in json.loads(f["items"]):
        valores.append((f["clip"], f"precio {item['producto']} ({item['tamano']})", item["precio_unitario"]))
    for n in re.findall(r"\d+", f["direccion_entrega"]):
        valores.append((f["clip"], "direccion_entrega", n))

tokens = {clip: set(normalizar_texto(texto).split()) for clip, texto in referencias.items()}
literal = pd.DataFrame(
    [{"clip": c, "campo": campo, "valor_gt": como_token(v), "aparece": como_token(v) in tokens[c]}
     for c, campo, v in valores]
)
literal.groupby("clip").aparece.agg(revisados="count", aparecen="sum").T

clip,e1,e2,e3,e4,e5,l1,l2,l3,l4,l5
revisados,2,2,2,2,2,7,6,7,9,8
aparecen,2,2,2,1,2,7,6,7,8,4


Los valores del ground truth que **no** aparecen literalmente:

In [9]:
literal[~literal.aparece]

,clip,campo,valor_gt,aparece
6,e4,anios_experiencia,2.5,False
35,l4,precio anticuchos (porción),18,False
41,l5,precio pollo a la brasa (entero),68,False
42,l5,precio pollo a la brasa (medio),36,False
43,l5,precio papas fritas (familiar),14,False
44,l5,precio gaseosa (1.5 litros),9,False


**Qué dicen estos ausentes**

- **E4 · anios_experiencia = 2.5.** Nunca se dice: el candidato cuenta "un año y medio" (1.5) y "otro año". La normalización produce `1.5`, no `2.5`. La extracción puede acertar sumando, pero la regla de verificación debe dejar el campo `sin_verificar`. Esta es la trampa que se plantó en E4.
- **L5 · todos los precios.** El vendedor nunca dicta los precios en esta llamada, y en L4 no dice el de los anticuchos. El precio unitario sale de la carta, no del audio. La verificación de `items` tiene que contrastar los precios con `carta.csv`, porque contra la transcripción nunca podría verificarlos.

**Lo que esta prueba no garantiza.** Que un número aparezca no prueba que ocupe el lugar correcto. En E3 el `8` de la experiencia está, pero también está `9` ("casi nueve"). En L3 aparecen `1043` y `1034`. Elegir cuál es el valor es trabajo de la extracción; esta capa solo garantiza que, si se dijo, se puede encontrar.

### 1.5 Formato de Whisper: casos escritos antes de la corrección

Con audio real (`06_resultados`, sección 2.5) aparecieron fallos que no vienen de lo que Whisper oye sino de **cómo lo escribe**: "6.500" con separador de miles, "$1.800" con símbolo, "semi-senior" con guion y "Calle Los Cedros, 245" con coma. Estos casos se escribieron, con su salida esperada, **antes** de cambiar `src/normalizacion.py`, y no se ajustaron después de ver el resultado. Son entradas nuevas: no copian frases del corpus.

- `casos_formato`: lo que debe dar `palabras_a_numeros`, que es lo que usa la extracción.
- `limitaciones_formato`: el caso ambiguo del separador de miles, con su salida ideal y la que se obtiene.
- `pares_formato`: lo que debe dar `normalizar_texto`, que es lo que usan la verificación y el WER.

In [10]:
casos_formato = [
    # grupo, entrada, esperado, motivo
    ("miles", "mi pretensión es 6.500 soles", "mi pretensión es 6500 soles", "punto + tres dígitos: miles (E1)"),
    ("miles", "idealmente 9.500.", "idealmente 9500.", "el punto final de la oración no es separador (E3)"),
    ("miles", "un presupuesto de 1.000.000", "un presupuesto de 1000000", "varios grupos de miles"),
    ("miles", "tengo 4.5 años", "tengo 4.5 años", "un dígito tras el punto: decimal"),
    ("miles", "unos 4,500 soles", "unos 4500 soles", "coma + tres dígitos: la misma regla que ya aplicaba normalizar_texto"),
    ("moneda", "alrededor de $1.800 al mes", "alrededor de 1800 dólares al mes", "símbolo de dólar pegado (E2)"),
    ("moneda", "S/ 6500", "6500 soles", "símbolo del sol"),
    ("moneda", "S/.4.800", "4800 soles", "forma antigua del sol, con separador de miles"),
    ("moneda", "US$ 2.000", "2000 dólares", "dólar explícito"),
    ("moneda", "$1.500 dólares", "1500 dólares", "si la moneda ya se dice detrás, no se repite"),
    ("moneda", "cuesta $ 20", "cuesta 20 dólares", "símbolo separado por un espacio"),
    ("guion", "semi-senior", "semi senior", "guion entre letras: espacio (E1)"),
    ("guion", "front-end", "front end", "guion entre letras: espacio (E4)"),
    ("guion", "en 2-3 semanas", "en 2-3 semanas", "entre cifras no es palabra compuesta: no se toca"),
]

limitaciones_formato = [
    # grupo, entrada, esperado (ideal), queda (lo que hace), motivo
    ("miles ambiguo", "pesa 2.125 kilos", "pesa 2.125 kilos", "pesa 2125 kilos",
     "un decimal con exactamente tres cifras se lee como miles; en el corpus no hay decimales de tres cifras"),
]

pares_formato = [
    # referencia, variante ASR, deben_coincidir, motivo
    ("mil ochocientos dólares al mes", "$1.800 al mes", True, "símbolo de dólar (E2)"),
    ("seis mil quinientos soles", "S/ 6.500", True, "símbolo del sol y separador de miles"),
    ("semi senior", "semi-senior", True, "guion entre letras"),
    ("Desarrollador frontend junior", "Desarrollador front-end junior", False,
     "el guion equivale al espacio, no a la palabra unida: 'front end' no es 'frontend'"),
]

tabla_formato = pd.DataFrame(
    [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e), "limitacion": False, "motivo": m}
     for g, e, esp, m in casos_formato]
    + [{"grupo": g, "entrada": e, "esperado": esp, "obtenido": palabras_a_numeros(e), "limitacion": True, "motivo": m}
       for g, e, esp, _queda, m in limitaciones_formato]
)
tabla_formato.insert(4, "pasa", tabla_formato.obtenido == tabla_formato.esperado)
pares_tabla = pd.DataFrame(
    [{"referencia": r, "variante_asr": v, "ref_normalizada": normalizar_texto(r), "asr_normalizada": normalizar_texto(v),
      "deben_coincidir": d, "motivo": m} for r, v, d, m in pares_formato]
)
pares_tabla.insert(4, "coinciden", pares_tabla.ref_normalizada == pares_tabla.asr_normalizada)
print(f"palabras_a_numeros: {tabla_formato.pasa.sum()} de {len(tabla_formato)} pasan · "
      f"normalizar_texto: {(pares_tabla.coinciden == pares_tabla.deben_coincidir).sum()} de {len(pares_tabla)} pares como se espera")
display(tabla_formato)
pares_tabla

palabras_a_numeros: 14 de 15 pasan · normalizar_texto: 4 de 4 pares como se espera


,grupo,entrada,esperado,obtenido,pasa,limitacion,motivo
0,miles,mi pretensión es 6.500 soles,mi pretensión es 6500 soles,mi pretensión es 6500 soles,True,False,punto + tres dígitos: miles (E1)
1,miles,idealmente 9.500.,idealmente 9500.,idealmente 9500.,True,False,el punto final de la oración no es separador (E3)
2,miles,un presupuesto de 1.000.000,un presupuesto de 1000000,un presupuesto de 1000000,True,False,varios grupos de miles
3,miles,tengo 4.5 años,tengo 4.5 años,tengo 4.5 años,True,False,un dígito tras el punto: decimal
4,miles,"unos 4,500 soles",unos 4500 soles,unos 4500 soles,True,False,coma + tres dígitos: la misma regla que ya aplicaba normalizar_texto
5,moneda,alrededor de $1.800 al mes,alrededor de 1800 dólares al mes,alrededor de 1800 dólares al mes,True,False,símbolo de dólar pegado (E2)
6,moneda,S/ 6500,6500 soles,6500 soles,True,False,símbolo del sol
7,moneda,S/.4.800,4800 soles,4800 soles,True,False,"forma antigua del sol, con separador de miles"
8,moneda,US$ 2.000,2000 dólares,2000 dólares,True,False,dólar explícito
9,moneda,$1.500 dólares,1500 dólares,1500 dólares,True,False,"si la moneda ya se dice detrás, no se repite"


,referencia,variante_asr,ref_normalizada,asr_normalizada,coinciden,deben_coincidir,motivo
0,mil ochocientos dólares al mes,$1.800 al mes,1800 dólares al mes,1800 dólares al mes,True,True,símbolo de dólar (E2)
1,seis mil quinientos soles,S/ 6.500,6500 soles,6500 soles,True,True,símbolo del sol y separador de miles
2,semi senior,semi-senior,semi senior,semi senior,True,True,guion entre letras
3,Desarrollador frontend junior,Desarrollador front-end junior,desarrollador frontend junior,desarrollador front end junior,False,False,"el guion equivale al espacio, no a la palabra unida: 'front end' no es 'frontend'"


In [11]:
for grupo, entrada, esperado, motivo in casos_formato:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == esperado, f"[{grupo}] {entrada!r}: se esperaba {esperado!r}, salió {obtenido!r}"
for grupo, entrada, esperado, queda, motivo in limitaciones_formato:
    obtenido = palabras_a_numeros(entrada)
    assert obtenido == queda and obtenido != esperado, f"[{grupo}] la limitación cambió: {entrada!r} da {obtenido!r}"
for r, v, d, m in pares_formato:
    assert (normalizar_texto(r) == normalizar_texto(v)) == d, f"{r!r} / {v!r}: {m}"
print(f"OK: {len(casos_formato)} casos, {len(limitaciones_formato)} limitación y {len(pares_formato)} pares de formato")

OK: 14 casos, 1 limitación y 4 pares de formato


Con las reglas de la versión 1 pasaban 3 de los 15 casos y 2 de los 4 pares (commit `2503caa`, donde se escribieron). Con la corrección de `src/normalizacion.py` (paso 0 del docstring y `config.yaml → normalizacion`) pasan los 14 casos y los 4 pares, y la limitación se comporta como está documentada.

- **Por qué `$` es dólar.** El corpus es de Lima: en Perú el sol se escribe `S/` y `$` se usa para el dólar, y Whisper escribe `$` donde se dice "dólares" (E2). En México o Chile `$` sería el peso: la tabla `simbolos_moneda` es de este corpus.
- **El caso ambiguo.** Whisper escribe los decimales con punto ("4.5 años") y también los miles ("6.500"), así que el punto solo no decide. La regla mira cuántos dígitos siguen: tres, miles; otro número, decimal. Un decimal de tres cifras ("2.125 kilos") se lee mal, pero en el corpus no hay ninguno.
- **`front-end` contra `frontend`.** El guion equivale al espacio, que es la forma de "semi senior" en el ground truth. El de E4 escribe "frontend" junto, y esa forma no la cubre esta regla: el par queda distinto a propósito.

## 2. Transcripciones y ground truth

> **Esto es el techo, con transcripción perfecta.** Todavía no hay transcripciones del audio, así que las secciones 2 a 7 usan las referencias de `data/referencias/` (el texto del guion) como si fueran la salida del ASR. Con audio real, el resultado solo puede empeorar. Además, las reglas se escribieron mirando estos mismos 10 textos, así que incluso como techo es optimista. La sección 3.2 lo mide con variantes que no son del corpus.

Cada referencia se aplana a un solo texto, sin saltos de línea ni etiquetas de hablante, porque así llega de Whisper. Las reglas no pueden saber quién habla.

In [12]:
from src.extraccion_reglas import extraer
from src.metricas import (aciertos, error_absoluto, ground_truth, intervalo_bootstrap, intervalo_wilson,
                          macro_f1)
from src.verificacion import verificar

ESQUEMA = {"e": "entrevista", "l": "pedido"}
transcripciones = {clip: " ".join(texto.split()) for clip, texto in referencias.items()}
gt = {**ground_truth("entrevista"), **ground_truth("pedido")}
print(f"{len(transcripciones)} transcripciones · {len(gt)} registros de ground truth")
print(transcripciones["l3"][:300], "…")

10 transcripciones · 10 registros de ground truth
La Leña Dorada, buenas noches. Hola, buenas noches. Quiero hacer un pedido: somos tres personas y cada uno quiere su combo. Claro. ¿Qué le preparo? Tres cuartos de pollo, cada uno con su papa personal. Tres cuartos de pollo a veinte soles cada uno, y tres papas personales a siete. ¿Bebidas? Tres gas …


## 3. Extracción por reglas

`src/extraccion_reglas.extraer(transcript, schema)` devuelve, para cada campo, `valor` y `evidencia`: la oración original de la que salió el valor. Las reglas están documentadas en el docstring del módulo y los marcadores en `config.yaml → extraccion.reglas`.

### 3.1 Valor extraído contra ground truth

In [13]:
def mostrar(valor):
    """Forma compacta de un valor para las tablas."""
    if isinstance(valor, list) and valor and isinstance(valor[0], dict):
        return "; ".join(f"{i['cantidad']}× {i['producto']} {i['tamano']} @{i['precio_unitario']}" for i in valor)
    if isinstance(valor, dict):
        return f"{valor['monto']} {valor['moneda']}"
    if isinstance(valor, list):
        return ", ".join(valor)
    return valor


extraidos = {clip: extraer(texto, ESQUEMA[clip[0]]) for clip, texto in transcripciones.items()}
filas = []
for clip, campos in extraidos.items():
    acierta = aciertos({k: v["valor"] for k, v in campos.items()}, gt[clip], ESQUEMA[clip[0]])
    filas += [{"clip": clip, "campo": campo, "extraido": mostrar(dato["valor"]),
               "ground_truth": mostrar(gt[clip][campo]), "acierta": acierta[campo]}
              for campo, dato in campos.items()]
resultado = pd.DataFrame(filas)
print(f"{resultado.acierta.sum()} de {len(resultado)} campos aciertan")
resultado.pivot(index="campo", columns="clip", values="acierta").reindex(
    [*cfg["esquemas"]["entrevista"]["campos"], *cfg["esquemas"]["pedido"]["campos"]])

60 de 60 campos aciertan

clip,e1,e2,e3,e4,e5,l1,l2,l3,l4,l5
campo,,,,,,,,,,
anios_experiencia,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
ultimo_puesto,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
tecnologias,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
pretension_salarial,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
disponibilidad,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
nivel_ingles,True,True,True,True,True,NaN,NaN,NaN,NaN,NaN
items,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True
promocion,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True
descuento,NaN,NaN,NaN,NaN,NaN,True,True,True,True,True


In [14]:
resultado

,clip,campo,extraido,ground_truth,acierta
0,e1,anios_experiencia,4.5,4.5,True
1,e1,ultimo_puesto,Desarrollador backend semi senior,Desarrollador backend semi senior,True
2,e1,tecnologias,"Python, Django, PostgreSQL, Docker","Python, Django, PostgreSQL, Docker",True
3,e1,pretension_salarial,6500 PEN,6500 PEN,True
4,e1,disponibilidad,2026-11-14,2026-11-14,True
5,e1,nivel_ingles,intermedio,intermedio,True
6,e2,anios_experiencia,3,3,True
7,e2,ultimo_puesto,Analista de datos senior,Analista de datos senior,True
8,e2,tecnologias,"Power BI, SQL Server, Python, pandas, Airflow","Power BI, SQL Server, Python, pandas, Airflow",True
9,e2,pretension_salarial,1800 USD,1800 USD,True


### 3.2 Variantes que no son del corpus

Para medir si las reglas sirven más allá de estos 10 textos, escribí 16 frases con otra redacción, **antes** de pasarlas por las reglas, y no cambié ninguna regla después de verlas. Las que fallan se dejan fallar. Siguen siendo frases de una sola persona, así que solo sirven como indicio, no como prueba.

In [15]:
tipos = {c: t for e in cfg["esquemas"].values() for c, t in e["campos"].items()}
variantes = [
    # id, esquema, campo, texto, esperado
    ("v01", "entrevista", "pretension_salarial",
     "¿Cuál es tu expectativa económica? Hoy cobro tres mil. Me gustaría llegar a cuatro mil quinientos soles.",
     {"monto": 4500, "moneda": "PEN"}),
    ("v02", "entrevista", "pretension_salarial",
     "¿Cuánto pides de sueldo? Pido dos mil dólares mensuales.", {"monto": 2000, "moneda": "USD"}),
    ("v03", "entrevista", "anios_experiencia",
     "¿Cuántos años de experiencia tienes? Llevo siete años en el rubro.", 7),
    ("v04", "entrevista", "anios_experiencia",
     "Háblame de tu experiencia. Trabajo en desarrollo desde dos mil diecinueve.", 7),
    ("v05", "entrevista", "disponibilidad",
     "¿Cuándo podrías empezar? Podría incorporarme en tres semanas.", "2026-11-21"),
    ("v06", "entrevista", "disponibilidad",
     "¿Desde cuándo estarías disponible? A partir del quince de noviembre.", "2026-11-15"),
    ("v07", "entrevista", "disponibilidad",
     "¿Cuándo puedes empezar? Necesito dos meses de preaviso.", "2026-12-31"),
    ("v08", "entrevista", "nivel_ingles", "¿Y tu inglés? Tengo un B2 certificado.", "intermedio"),
    ("v09", "entrevista", "tecnologias",
     "Uso Java con Spring Boot. Antes usaba Angular, pero lo reemplazamos por React.",
     ["Java", "Spring Boot", "React"]),
    ("v10", "entrevista", "ultimo_puesto",
     "Actualmente soy ingeniera de datos en una aseguradora.", "Ingeniera de datos"),
    ("v11", "entrevista", "ultimo_puesto",
     "Me desempeño como arquitecto de soluciones desde hace tres años.", "Arquitecto de soluciones"),
    ("v12", "pedido", "items",
     "Buenas, quiero un pollo entero y dos gaseosas personales. Le repito su pedido: un pollo entero y dos gaseosas personales.",
     [{"producto": "pollo a la brasa", "tamano": "entero", "cantidad": 1, "precio_unitario": 68},
      {"producto": "gaseosa", "tamano": "personal", "cantidad": 2, "precio_unitario": 4}]),
    ("v13", "pedido", "items",
     "Quisiera una jarra de chicha morada y un cuarto de pollo. Mejor que sean dos cuartos de pollo.",
     [{"producto": "chicha morada", "tamano": "jarra", "cantidad": 1, "precio_unitario": 12},
      {"producto": "pollo a la brasa", "tamano": "cuarto", "cantidad": 2, "precio_unitario": 20}]),
    ("v14", "pedido", "direccion_entrega",
     "La dirección es Jr. Huallaga trescientos diez, piso dos, en Cercado de Lima.",
     "Jr. Huallaga 310, piso 2, Cercado de Lima"),
    ("v15", "pedido", "metodo_pago", "¿Cómo va a pagar? Con Yape, ¿o aceptan Plin?", "yape"),
    ("v16", "pedido", "total_dicho", "Serían ochenta soles en total.", 80),
]
from src.metricas import coincide

tabla_variantes = pd.DataFrame(
    [{"id": i, "campo": campo, "texto": texto, "esperado": mostrar(esp),
      "obtenido": mostrar(ob := extraer(texto, esquema)[campo]["valor"]),
      "acierta": coincide(tipos[campo], ob, esp)} for i, esquema, campo, texto, esp in variantes]
)
print(f"{tabla_variantes.acierta.sum()} de {len(tabla_variantes)} variantes aciertan")
tabla_variantes

12 de 16 variantes aciertan


,id,campo,texto,esperado,obtenido,acierta
0,v01,pretension_salarial,¿Cuál es tu expectativa económica? Hoy cobro tres mil. Me gustaría llegar a cuatro mil quinientos soles.,4500 PEN,4500 PEN,True
1,v02,pretension_salarial,¿Cuánto pides de sueldo? Pido dos mil dólares mensuales.,2000 USD,2000 USD,True
2,v03,anios_experiencia,¿Cuántos años de experiencia tienes? Llevo siete años en el rubro.,7,7,True
3,v04,anios_experiencia,Háblame de tu experiencia. Trabajo en desarrollo desde dos mil diecinueve.,7,7,True
4,v05,disponibilidad,¿Cuándo podrías empezar? Podría incorporarme en tres semanas.,2026-11-21,2026-11-21,True
5,v06,disponibilidad,¿Desde cuándo estarías disponible? A partir del quince de noviembre.,2026-11-15,2026-11-15,True
6,v07,disponibilidad,¿Cuándo puedes empezar? Necesito dos meses de preaviso.,2026-12-31,2026-12-31,True
7,v08,nivel_ingles,¿Y tu inglés? Tengo un B2 certificado.,intermedio,intermedio,True
8,v09,tecnologias,"Uso Java con Spring Boot. Antes usaba Angular, pero lo reemplazamos por React.","Java, Spring Boot, React","Java, Spring Boot",False
9,v10,ultimo_puesto,Actualmente soy ingeniera de datos en una aseguradora.,Ingeniera de datos,Ingeniera de datos,True


### 3.3 Items: confirmación del vendedor contra diálogo

La regla toma los items de la **última confirmación del vendedor** ("le confirmo", "le repito"). Esa confirmación ya contiene el pedido corregido, mientras que el diálogo trae cada versión intermedia ("dos medios pollos" → "mejor un entero"). Así, la autocorrección la resuelve el vendedor en voz alta y la regla no tiene que reconstruirla.

Para ver qué pasaría sin confirmación, se borra esa oración de cada llamada y se extrae otra vez con el camino de respaldo, que recorre el diálogo aplicando las correcciones.

In [16]:
comparacion_items = []
for clip in [c for c in transcripciones if c.startswith("l")]:
    sin_confirmacion = transcripciones[clip].replace(extraidos[clip]["items"]["evidencia"], "")
    respaldo = extraer(sin_confirmacion, "pedido")["items"]
    comparacion_items.append({
        "clip": clip,
        "confirmacion_acierta": coincide("lista_objetos", extraidos[clip]["items"]["valor"], gt[clip]["items"]),
        "fuente_sin_confirmacion": respaldo.get("fuente"),
        "dialogo_acierta": coincide("lista_objetos", respaldo["valor"], gt[clip]["items"]),
        "oraciones_usadas_por_el_dialogo": respaldo["evidencia"].count(" … ") + 1 if respaldo["evidencia"] else 0,
    })
pd.DataFrame(comparacion_items)

,clip,confirmacion_acierta,fuente_sin_confirmacion,dialogo_acierta,oraciones_usadas_por_el_dialogo
0,l1,True,dialogo,True,4
1,l2,True,dialogo,True,8
2,l3,True,dialogo,True,6
3,l4,True,dialogo,True,9
4,l5,True,dialogo,True,4


En estas 5 llamadas, con transcripción perfecta, **los dos caminos aciertan**, así que estos números no justifican por sí solos la decisión. Las razones para preferir la confirmación son otras:

- **Exposición al ASR.** La confirmación es una sola oración. El diálogo necesita entre 4 y 9 oraciones correctas, y en cada una el ASR puede equivocarse con un número o con un marcador de corrección ("no, perdón"). Si un marcador se pierde, el diálogo se queda con la versión anterior del pedido.
- **Quién resuelve la corrección.** En la confirmación es el vendedor el que dice el pedido final. En el diálogo, la regla tiene que adivinar qué reemplaza a qué.

La comparación de verdad llega con audio degradado, en `04_degradacion`.

### 3.4 Variantes de formato de Whisper

Igual que en la 3.2: 13 frases nuevas con la salida esperada, escritas **antes** de corregir las reglas y sin ajustarlas después. Esta vez cada una usa un formato que Whisper produjo con audio real (`06_resultados`, sección 2.5), en frases que no son del corpus. `f03`, `f05` y `f13` son controles: la versión 1 ya los resolvía y la corrección no debe romperlos.

In [17]:
variantes_formato = [
    # id, formato, esquema, campo, texto, esperado
    ("f01", "miles", "entrevista", "pretension_salarial",
     "¿Cuál es tu pretensión salarial? Pido 4.200 soles mensuales.", {"monto": 4200, "moneda": "PEN"}),
    ("f02", "miles", "entrevista", "pretension_salarial",
     "¿Y tu expectativa de sueldo? Hoy gano 3.900. Busco unos 4.600 soles.", {"monto": 4600, "moneda": "PEN"}),
    ("f03", "miles", "entrevista", "anios_experiencia",
     "¿Cuántos años de experiencia tienes? Tengo 3.5 años programando.", 3.5),
    ("f04", "moneda", "entrevista", "pretension_salarial",
     "¿Cuál sería tu expectativa? Algo como $2.300 al mes.", {"monto": 2300, "moneda": "USD"}),
    ("f05", "moneda", "entrevista", "pretension_salarial",
     "¿Cuál es tu pretensión? Mi pretensión es S/ 5200 mensuales.", {"monto": 5200, "moneda": "PEN"}),
    ("f06", "moneda", "entrevista", "pretension_salarial",
     "¿Cuánto buscas ganar? Estoy pidiendo S/.4.800 brutos.", {"monto": 4800, "moneda": "PEN"}),
    ("f07", "moneda", "entrevista", "pretension_salarial",
     "¿Cuál es tu pretensión salarial? Unos $1.500 dólares mensuales.", {"monto": 1500, "moneda": "USD"}),
    ("f08", "guion", "entrevista", "ultimo_puesto",
     "Actualmente soy analista semi-senior en un banco.", "Analista semi senior"),
    ("f09", "guion", "entrevista", "ultimo_puesto",
     "Trabajo como desarrolladora full-stack en una consultora.", "Desarrolladora full stack"),
    ("f10", "dirección", "pedido", "direccion_entrega",
     "La dirección es Avenida Arequipa, 1250, departamento 801, en Lince.", "Avenida Arequipa 1250, departamento 801, Lince"),
    ("f11", "dirección", "pedido", "direccion_entrega",
     "Mándelo a Jirón Ucayali, 315, interior 2, en Breña.", "Jirón Ucayali 315, interior 2, Breña"),
    ("f12", "dirección", "pedido", "direccion_entrega",
     "Es en Calle Las Begonias 1.430, en San Isidro.", "Calle Las Begonias 1430, San Isidro"),
    ("f13", "moneda", "pedido", "total_dicho", "El total con delivery es S/ 104.", 104),
]

tabla_formato_variantes = pd.DataFrame(
    [{"id": i, "formato": f, "campo": campo, "texto": texto, "esperado": mostrar(esp),
      "obtenido": mostrar(ob := extraer(texto, esquema)[campo]["valor"]),
      "acierta": coincide(tipos[campo], ob, esp)} for i, f, esquema, campo, texto, esp in variantes_formato]
)
print(f"{tabla_formato_variantes.acierta.sum()} de {len(tabla_formato_variantes)} variantes de formato aciertan")
tabla_formato_variantes

13 de 13 variantes de formato aciertan


,id,formato,campo,texto,esperado,obtenido,acierta
0,f01,miles,pretension_salarial,¿Cuál es tu pretensión salarial? Pido 4.200 soles mensuales.,4200 PEN,4200 PEN,True
1,f02,miles,pretension_salarial,¿Y tu expectativa de sueldo? Hoy gano 3.900. Busco unos 4.600 soles.,4600 PEN,4600 PEN,True
2,f03,miles,anios_experiencia,¿Cuántos años de experiencia tienes? Tengo 3.5 años programando.,3.5,3.5,True
3,f04,moneda,pretension_salarial,¿Cuál sería tu expectativa? Algo como $2.300 al mes.,2300 USD,2300 USD,True
4,f05,moneda,pretension_salarial,¿Cuál es tu pretensión? Mi pretensión es S/ 5200 mensuales.,5200 PEN,5200 PEN,True
5,f06,moneda,pretension_salarial,¿Cuánto buscas ganar? Estoy pidiendo S/.4.800 brutos.,4800 PEN,4800 PEN,True
6,f07,moneda,pretension_salarial,¿Cuál es tu pretensión salarial? Unos $1.500 dólares mensuales.,1500 USD,1500 USD,True
7,f08,guion,ultimo_puesto,Actualmente soy analista semi-senior en un banco.,Analista semi senior,Analista semi senior,True
8,f09,guion,ultimo_puesto,Trabajo como desarrolladora full-stack en una consultora.,Desarrolladora full stack,Desarrolladora full stack,True
9,f10,dirección,direccion_entrega,"La dirección es Avenida Arequipa, 1250, departamento 801, en Lince.","Avenida Arequipa 1250, departamento 801, Lince","Avenida Arequipa 1250, departamento 801, Lince",True


In [18]:
assert tabla_formato_variantes.acierta.all(), tabla_formato_variantes[~tabla_formato_variantes.acierta]
fallan = set(tabla_variantes.loc[~tabla_variantes.acierta, "id"])
assert fallan == {"v09", "v14", "v15", "v16"}, f"cambiaron los fallos de la 3.2: {sorted(fallan)}"
print("OK: 13 de 13 variantes de formato; la 3.2 sigue en 12 de 16 con los mismos 4 fallos")

OK: 13 de 13 variantes de formato; la 3.2 sigue en 12 de 16 con los mismos 4 fallos


Con la versión 1 acertaban 3 de 13, justo los controles; con la versión 2 aciertan las 13. Las 16 variantes de la 3.2 dan lo mismo que antes: 12 de 16, con los mismos cuatro fallos (v09, v14, v15 y v16), que siguen sin corregirse. La v14 ("Jr. Huallaga") no la arregla la puntuación nueva: allí el problema es que la oración se corta en la abreviatura, antes de llegar a la regla de dirección.

Las variantes miden que la regla hace lo que dice en frases nuevas, no que cubra todo lo que Whisper puede escribir. El efecto sobre el audio real está en `06_resultados`, sección 5.

## 4. Extracción por LLM

JSON estructurado con el prompt de `config.yaml`. Cada valor viene con su evidencia literal.

*Pendiente.*

## 5. Verificación

`src/verificacion.verificar(campos, transcript)` asigna a cada valor uno de tres estados. Las listas se verifican elemento por elemento.

- **verificado**: el valor aparece literalmente en la transcripción, después de normalizar las dos.
- **verificado_carta**: solo para `precio_unitario`, que sale de la carta y no del audio. Se cuenta aparte.
- **sin_verificar**: no aparece, aunque el valor sea correcto.

### 5.1 Estado de cada campo, con su evidencia

In [19]:
verificados = {clip: verificar(extraidos[clip], transcripciones[clip]) for clip in transcripciones}

filas = []
for clip, estados in verificados.items():
    for campo, estado in estados.items():
        dato = extraidos[clip][campo]
        if campo == "tecnologias":
            filas += [{"clip": clip, "campo": f"tecnologias: {t}", "valor": t, "estado": e, "evidencia": ev}
                      for t, e, ev in zip(dato["valor"], estado, dato["evidencia"])]
        elif campo == "items":
            for item, e, mencion in zip(dato["valor"], estado, dato["menciones"]):
                nombre = f"{item['producto']} {item['tamano']}"
                filas.append({"clip": clip, "campo": f"item: {nombre}", "valor": f"{item['cantidad']}× {nombre}",
                              "estado": e["item"], "evidencia": mencion})
                filas.append({"clip": clip, "campo": f"precio: {nombre}", "valor": item["precio_unitario"],
                              "estado": e["precio_unitario"], "evidencia": "carta.csv"})
        else:
            filas.append({"clip": clip, "campo": campo, "valor": mostrar(dato["valor"]), "estado": estado,
                          "evidencia": dato["evidencia"]})
verificacion = pd.DataFrame(filas)
verificacion

,clip,campo,valor,estado,evidencia
0,e1,anios_experiencia,4.5,verificado,"Bueno, si cuento solo lo formal, con contrato, serían cuatro años y medio."
1,e1,ultimo_puesto,Desarrollador backend semi senior,verificado,Ahí estoy como desarrollador backend semi senior.
2,e1,tecnologias: Python,Python,verificado,"Principalmente Python, con Django para las APIs."
3,e1,tecnologias: Django,Django,verificado,"Principalmente Python, con Django para las APIs."
4,e1,tecnologias: PostgreSQL,PostgreSQL,verificado,"La base de datos es PostgreSQL, y todo lo desplegamos en contenedores con Docker."
...,...,...,...,...,...
107,l5,promocion,duo familiar,verificado,"Le confirmo: un pollo entero, un medio pollo, dos papas familiares y dos gaseosas de litro y medio, con el dúo familiar."
108,l5,descuento,10,verificado,"Le confirmo: un pollo entero, un medio pollo, dos papas familiares y dos gaseosas de litro y medio, con el dúo familiar."
109,l5,direccion_entrega,"Avenida Los Laureles 2271, bloque C, departamento 504, San Juan de Lurigancho",verificado,"Avenida Los Laureles dos mil doscientos setenta y uno, bloque C, departamento quinientos cuatro, en San Juan de Lurigancho."
110,l5,metodo_pago,yape,verificado,Yape.


### 5.2 Resumen por estado

In [20]:
verificacion.assign(grupo=verificacion.campo.str.split(":").str[0]).pivot_table(
    index="grupo", columns="estado", values="clip", aggfunc="count", fill_value=0)

estado,sin_verificar,verificado,verificado_carta
grupo,,,
anios_experiencia,1,4,0
descuento,0,5,0
direccion_entrega,0,5,0
disponibilidad,0,5,0
item,0,19,0
metodo_pago,0,5,0
nivel_ingles,0,5,0
precio,0,0,19
pretension_salarial,0,5,0


In [21]:
sin_verificar = verificacion[verificacion.estado == cfg["verificacion"]["etiqueta_no_verificado"]]
sin_verificar.merge(resultado.rename(columns={"acierta": "valor_correcto"})[["clip", "campo", "valor_correcto"]],
                    on=["clip", "campo"], how="left")

,clip,campo,valor,estado,evidencia,valor_correcto
0,e4,anios_experiencia,2.5,sin_verificar,"Un año y medio en mi primer trabajo, en una agencia, y otro año en el actual.",True


In [22]:
assert verificados["e4"]["anios_experiencia"] == "sin_verificar", "E4: 2.5 años nunca se dicen"
assert all(e["precio_unitario"] == "verificado_carta" for c in verificados if c.startswith("l")
           for e in verificados[c]["items"]), "todo precio correcto debe ser verificado_carta"
print("OK: E4 queda sin_verificar y todos los precios son verificado_carta")

OK: E4 queda sin_verificar y todos los precios son verificado_carta


Solo un valor queda `sin_verificar`, y es correcto:

- **E4 · anios_experiencia = 2.5.** Es la trampa plantada: el candidato dice "un año y medio" y "otro año". La regla suma y acierta, pero el 2.5 nunca se dice, y la verificación no lo da por bueno. Así debe ser: sumar es interpretar.
- **L3 · promocion = null (y descuento = 0)** queda `verificado`. No hay promoción, y la ausencia se dijo en voz alta: ante "¿Tiene algún código de promoción?", el cliente responde "No, ninguno.", y esa respuesta es la evidencia literal. Un campo nulo que nunca se menciona queda en otro estado, `no_mencionado`, para no contarlo como fallo de verificación (ver el docstring de `src/verificacion.py`).

Con la fecha la regla es distinta. "en 2 semanas" → 2026-11-14 queda `verificado`, porque la expresión aparece en el texto y, interpretada desde la fecha de grabación, da exactamente la fecha extraída. Es una conversión de lo dicho, igual que "seis mil quinientos" → 6500. El 2.5 de E4, en cambio, combina dos cosas dichas.

## 6. Capa de transcripción

WER y CER por clip.

*Pendiente.*

## 7. Capa de registro

Exactitud por campo con dos intervalos al 95 %, macro F1 en `nivel_ingles` y error absoluto en los campos numéricos.

> **Con 5 clips por tarea, cualquier intervalo es muy ancho.** Un campo que acierta 5 de 5 tiene un intervalo de Wilson de [0.57, 1]: la muestra es compatible con un sistema que falla en 4 de cada 10 casos. El bootstrap de 5 de 5 da [1, 1] porque no puede remuestrear un error que no vio. **Ese intervalo no expresa certeza, sino falta de datos**; por eso se reporta también Wilson.

In [23]:
filas = []
for esquema in ("entrevista", "pedido"):
    clips = [c for c in transcripciones if ESQUEMA[c[0]] == esquema]
    for campo in cfg["esquemas"][esquema]["campos"]:
        valores = [int(aciertos({k: v["valor"] for k, v in extraidos[c].items()}, gt[c], esquema)[campo]) for c in clips]
        filas.append({"esquema": esquema, "campo": campo, "aciertos": sum(valores), "n": len(valores),
                      "exactitud": sum(valores) / len(valores),
                      "ic95_wilson": tuple(round(x, 2) for x in intervalo_wilson(sum(valores), len(valores))),
                      "ic95_bootstrap": tuple(round(x, 2) for x in intervalo_bootstrap(valores))})
pd.DataFrame(filas)

,esquema,campo,aciertos,n,exactitud,ic95_wilson,ic95_bootstrap
0,entrevista,anios_experiencia,5,5,1.0,"(0.57, 1.0)","(1, 1)"
1,entrevista,ultimo_puesto,5,5,1.0,"(0.57, 1.0)","(1, 1)"
2,entrevista,tecnologias,5,5,1.0,"(0.57, 1.0)","(1, 1)"
3,entrevista,pretension_salarial,5,5,1.0,"(0.57, 1.0)","(1, 1)"
4,entrevista,disponibilidad,5,5,1.0,"(0.57, 1.0)","(1, 1)"
5,entrevista,nivel_ingles,5,5,1.0,"(0.57, 1.0)","(1, 1)"
6,pedido,items,5,5,1.0,"(0.57, 1.0)","(1, 1)"
7,pedido,promocion,5,5,1.0,"(0.57, 1.0)","(1, 1)"
8,pedido,descuento,5,5,1.0,"(0.57, 1.0)","(1, 1)"
9,pedido,direccion_entrega,5,5,1.0,"(0.57, 1.0)","(1, 1)"


In [24]:
clases = cfg["esquemas"]["entrevista"]["clases"]["nivel_ingles"]
entrevistas_clips = [c for c in transcripciones if c.startswith("e")]
pares = [(gt[c]["nivel_ingles"], extraidos[c]["nivel_ingles"]["valor"]) for c in entrevistas_clips]
f1 = macro_f1(*zip(*pares), clases)
ic = intervalo_bootstrap(pares, lambda m: macro_f1(*zip(*m), clases))
print(f"macro F1 nivel_ingles = {f1:.2f}   IC95 bootstrap = [{ic[0]:.2f}, {ic[1]:.2f}]")
print("Distribución real:", pd.Series([v for v, _ in pares]).value_counts().to_dict())
print("Si el único 'basico' (E3) se predijera 'intermedio':",
      round(macro_f1([v for v, _ in pares], ["intermedio" if c == "e3" else p for c, (_, p) in zip(entrevistas_clips, pares)], clases), 2))

macro F1 nivel_ingles = 1.00   IC95 bootstrap = [1.00, 1.00]
Distribución real: {'intermedio': 2, 'avanzado': 2, 'basico': 1}
Si el único 'basico' (E3) se predijera 'intermedio': 0.6


In [25]:
numericos = {
    "anios_experiencia": lambda r: r["anios_experiencia"],
    "pretension_salarial (monto)": lambda r: r["pretension_salarial"]["monto"] if r["pretension_salarial"] else None,
    "total_dicho": lambda r: r["total_dicho"],
    "descuento": lambda r: r["descuento"],
}
filas = []
for nombre, valor in numericos.items():
    clips = [c for c in transcripciones if nombre.split()[0] in gt[c]]
    errores = error_absoluto([valor(gt[c]) for c in clips], [valor({k: v["valor"] for k, v in extraidos[c].items()}) for c in clips])
    medidos = [e for e in errores if e is not None]
    filas.append({"campo": nombre, "n": len(clips), "sin_prediccion": errores.count(None),
                  "error_absoluto_medio": sum(medidos) / len(medidos) if medidos else None,
                  "maximo": max(medidos) if medidos else None,
                  "ic95_bootstrap": tuple(intervalo_bootstrap(medidos)) if medidos else None})
pd.DataFrame(filas)

,campo,n,sin_prediccion,error_absoluto_medio,maximo,ic95_bootstrap
0,anios_experiencia,5,0,0.0,0.0,"(0.0, 0.0)"
1,pretension_salarial (monto),5,0,0.0,0.0,"(0.0, 0.0)"
2,total_dicho,5,0,0.0,0.0,"(0.0, 0.0)"
3,descuento,5,0,0.0,0.0,"(0.0, 0.0)"


### 7.1 Fallos y su causa

**En las 10 referencias del corpus.** Los 60 campos aciertan. El único `sin_verificar` (E4 `anios_experiencia`) no es un fallo de extracción: el valor es correcto y la verificación lo marca como no comprobable (sección 5). Que todo acierte no prueba que las reglas generalicen: se escribieron mirando estos mismos textos, y por eso existe la sección 3.2.

**En las variantes que no son del corpus (sección 3.2).** Fallan 4 de 16, y se dejan fallar:

| Variante | Campo | Causa | Qué haría falta |
| --- | --- | --- | --- |
| v09 | tecnologias | El abandono se decide por oración completa: "Antes usaba Angular, pero lo reemplazamos por React" descarta también React, que es la tecnología actual. | Decidir el abandono por término, no por oración (qué queda a cada lado de "reemplazamos por"). |
| v14 | direccion_entrega | "Jr." se lee como fin de oración: la oración se corta en la abreviatura y el tipo de vía queda separado del nombre. | No cortar oraciones tras abreviaturas conocidas (Jr., Av., Sr.). |
| v15 | metodo_pago | La respuesta "Con Yape, ¿o aceptan Plin?" termina en "?" y se toma como pregunta. Sin respuesta reconocible, vale el último método del texto: Plin. | Sin hablantes, "?" es una aproximación del cambio de turno; con diarización se resolvería. |
| v16 | total_dicho | La regla solo busca el número **después** de "total"; "ochenta soles en total" lo dice antes. | Añadir el patrón "N soles en total". |

Ninguna se corrigió después de verla, para que la sección 3.2 siga midiendo algo. Son candidatas para la siguiente iteración de reglas, y cualquier corrección debería probarse con variantes nuevas.

## 8. ¿El WER predice si el registro sirve?

Un clip con WER alto y registro perfecto, y uno con WER bajo y registro roto.

*Pendiente.*

## 9. Reglas contra LLM

Comparación en los mismos clips.

*Pendiente.*